In [82]:
import pandas as pd
import glob, os
from tqdm.notebook import tqdm
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
import chart_studio
import chart_studio.plotly as py
import plotly.io as pio
import re, string, unicodedata
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', -1)


In [83]:
username = 'politicai' # your username
api_key = 'REDACTED' # your api key - go to profile > settings > regenerate key
chart_studio.tools.set_credentials_file(username=username, api_key=api_key)
# py.plot(fig_object, filename = 'gdp_per_cap', auto_open=True)
# pio.write_html(fig, file=’index.html’, auto_open=True)

In [84]:
def clean_df_format(df):
    df.columns = [x.lower().replace("/", "_").replace(" ", "_").replace("(", "").replace(")", "").strip() for x in df.columns]
    return df

def clean_country_names(df, column, new_country_col_name="country"):
    df[column] = df[column].str.lower()
    df[column] = df[column].str.replace("(", "").str.replace(")", "").str.replace("&", "")
    df[column] = df[column].str.replace(".", "").str.replace(",", "").str.replace("'","")
    df[column] = df[column].str.replace("-", " ").str.replace(" ", "_").str.replace("__", "_")
    
    name_changes={"korea_north":"north_korea", "korea_dem_people’s_rep":"north_korea","korea_south":"south_korea",
                  "korea_rep":"south_korea" , "korea_dem_peoples_republic_of":"north_korea", 
                  "korea_republic_of":"south_korea", "moldova_republic_of":"moldova",
                  "cape_verde":"cabo_verde", "syrian_arab_republic":"syria", "venezuela_rb":"venezuela",
                 "macedonia":"north_macedonia", "serbia_montenegro":"serbia", "slovak_republic":"slovakia",
                 "dem_rep_of_congo":"congo_kinshasa", "congo_dem_rep":"congo_kinshasa","congo_rep":"congo_brazzaville",
                  "congo_democratic_republic_of":"congo_kinshasa", "republic_of_the_congo_brazzaville": "congo_brazzaville",
                   "tanzania_united_republic_of":"tanzania", "north_macedonia_republic_of_":"north_macedonia",
                  "egypt_arab_rep":"egypt", "gambia_the":"gambia","the_gambia":"gambia" ,"russian_federation":"russia",
                 "iran_islamic_rep":"iran", "iran_islamic_republic_of":"iran",
                  "yemen_rep":"yemen", "kyrgyz_republic":"kyrgyzstan", "viet_nam":"vietnam",
                 "lao_pdr":"laos", "bahamas_the":"bahamas", "brunei_darussalam":"brunei", "sudan_north":"sudan", 
                 "hong_kong_sar_china":"hong_kong", "lao_peoples_dem_republic":"laos"}
    df[column] =  df[column].replace(name_changes)

    df = df.rename(columns={column:new_country_col_name})
    return df

# Cargar ficheros

In [85]:
df_fragility = pd.read_excel("../../data/staging/mundial/fragilidad.xls")
df_fragility = clean_df_format(df_fragility)

In [86]:
df_freedomhouse = pd.read_excel("../../data/staging/mundial/freedom_house.xlsx")
df_freedomhouse = clean_df_format(df_freedomhouse)

In [87]:
df_poblacionactiva = pd.read_excel("../../data/staging/mundial/tasa_poblacion_activa.xls")
df_poblacionactiva = clean_df_format(df_poblacionactiva)

In [88]:
df_esperanzavida = pd.read_excel("../../data/staging/mundial/esperanza_vida.xls")
df_esperanzavida = clean_df_format(df_esperanzavida)

In [89]:
df_escolarizacion = pd.read_csv("../../data/staging/mundial/años_escolarizacion_medios.csv", sep=",")
df_escolarizacion = clean_df_format(df_escolarizacion)

In [90]:
df_sistemaselectorales = pd.read_excel("../../data/staging/mundial/sistemas_electorales.xls")
df_sistemaselectorales = clean_df_format(df_sistemaselectorales)

In [91]:
df_poblacion = pd.read_excel("../../data/staging/mundial/poblacion.xls")
df_poblacion = clean_df_format(df_poblacion)

In [92]:
# escribir
# COWCODE, COUNTRY,YEAR,HOSELECT,HOSTERM,HOSDISS,HOGELECT,HOGTERM,HOGDISS,LEGISL,HOUSENUM,FEDUNIT
#df_constituciones = pd.read_excel("../../data/staging/mundial/constituciones.xlsx")
#df_constituciones = df_constituciones[["cowcode","country","year","hoselect","hosterm","hosdiss","hogelect","hogterm","hogdiss","legisl","housenum","fedunit"]]
#df_constituciones.to_csv("../../data/staging/mundial/constituciones_reducido.csv", sep=",", index=False)
# cargar
df_constituciones_red = pd.read_csv("../../data/staging/mundial/constituciones_reducido.csv", sep=",")
df_constituciones_red = clean_df_format(df_constituciones_red)

In [93]:
df_cow = pd.read_csv("../../data/staging/mundial/cow_country_codes.csv", sep=",")
df_cow = clean_df_format(df_cow)

# Tabla Macro

## Fragility

In [94]:
df_base = df_fragility[["scode", "country", "year", "sfi"]]
df_base = df_base.rename(columns={"scode":"code", "sfi":"state_fragility_idx"})
df_base = clean_country_names(df=df_base, column="country")
print("paises: ", len(df_base.country.unique()))
print("filas: ", df_base.shape)
df_base.head()

paises:  167
filas:  (3960, 4)


,code,country,year,state_fragility_idx
0,AFG,afghanistan,1995,25
1,AFG,afghanistan,1996,24
2,AFG,afghanistan,1997,24
3,AFG,afghanistan,1998,24
4,AFG,afghanistan,1999,24


### Freedom in the world

In [95]:
df_freedomhouse_idx = df_freedomhouse[["country_territory", "status", "total", "edition"]]
df_freedomhouse_idx = df_freedomhouse_idx.rename(columns={"status":"free_not_free", "total":"democracy_idx",
                                                         "edition":"year"})
df_freedomhouse_idx = clean_country_names(df=df_freedomhouse_idx, column="country_territory")
df_freedomhouse_idx.head()

,country,free_not_free,democracy_idx,year
0,abkhazia,PF,40,2020
1,afghanistan,NF,27,2020
2,albania,PF,67,2020
3,algeria,NF,34,2020
4,andorra,F,94,2020


In [96]:
df_base = df_base.merge(right=df_freedomhouse_idx, on=["country", "year"], how="inner")
print("paises: ", len(df_base.country.unique()))
print("filas: ", df_base.shape)
df_base.head()

paises:  166
filas:  (996, 6)


,code,country,year,state_fragility_idx,free_not_free,democracy_idx
0,AFG,afghanistan,2013,21,NF,27
1,AFG,afghanistan,2014,21,NF,26
2,AFG,afghanistan,2015,21,NF,24
3,AFG,afghanistan,2016,21,NF,24
4,AFG,afghanistan,2017,21,NF,24


### Poblacion activa

In [97]:
df_poblacionactiva_idx = pd.melt(df_poblacionactiva, id_vars=['country_name','country_code'], value_vars=df_poblacionactiva.columns[4:])
df_poblacionactiva_idx = df_poblacionactiva_idx[["country_name", "country_code", "variable", "value"]]
df_poblacionactiva_idx = df_poblacionactiva_idx.rename(columns={"country_code":"code_wb", "variable":"year",
                                                         "value":"tasa_pob_activa"})
df_poblacionactiva_idx = clean_country_names(df=df_poblacionactiva_idx, column="country_name")
df_poblacionactiva_idx.year = df_poblacionactiva_idx.year.astype(int)

df_poblacionactiva_idx.head()

,country,code_wb,year,tasa_pob_activa
0,aruba,ABW,1960,NaN
1,afghanistan,AFG,1960,NaN
2,angola,AGO,1960,NaN
3,albania,ALB,1960,NaN
4,andorra,AND,1960,NaN


In [98]:
df_base = df_base.merge(right=df_poblacionactiva_idx, on=["country", "year"], how="inner")
print("paises: ", len(df_base.country.unique()))
print("filas: ", df_base.shape)
df_base.head()

paises:  165
filas:  (990, 8)


,code,country,year,state_fragility_idx,free_not_free,democracy_idx,code_wb,tasa_pob_activa
0,AFG,afghanistan,2013,21,NF,27,AFG,48.016998
1,AFG,afghanistan,2014,21,NF,26,AFG,48.188999
2,AFG,afghanistan,2015,21,NF,24,AFG,48.374001
3,AFG,afghanistan,2016,21,NF,24,AFG,48.516998
4,AFG,afghanistan,2017,21,NF,24,AFG,48.666000


### Esperanza de vida

In [99]:
df_esperanzavida_idx = pd.melt(df_esperanzavida, id_vars=['country_name','country_code'], value_vars=df_esperanzavida.columns[4:])
df_esperanzavida_idx = df_esperanzavida_idx[["country_code", "variable", "value"]]
df_esperanzavida_idx = df_esperanzavida_idx.rename(columns={"country_code":"code_wb", "variable":"year",
                                                         "value":"esperanza_vida"})
df_esperanzavida_idx.year = df_esperanzavida_idx.year.astype(int)

df_esperanzavida_idx.head()

,code_wb,year,esperanza_vida
0,ABW,1960,65.662
1,AFG,1960,32.446
2,AGO,1960,37.524
3,ALB,1960,62.283
4,AND,1960,NaN


In [100]:
df_base = df_base.merge(right=df_esperanzavida_idx, on=["code_wb", "year"], how="inner")
print("paises: ", len(df_base.country.unique()))
print("filas: ", df_base.shape)
df_base.head()

paises:  165
filas:  (990, 9)


,code,country,year,state_fragility_idx,free_not_free,democracy_idx,code_wb,tasa_pob_activa,esperanza_vida
0,AFG,afghanistan,2013,21,NF,27,AFG,48.016998,62.525
1,AFG,afghanistan,2014,21,NF,26,AFG,48.188999,62.966
2,AFG,afghanistan,2015,21,NF,24,AFG,48.374001,63.377
3,AFG,afghanistan,2016,21,NF,24,AFG,48.516998,63.763
4,AFG,afghanistan,2017,21,NF,24,AFG,48.666000,64.130


### Años escolarización

In [101]:
df_escolarizacion_idx = df_escolarizacion
df_escolarizacion_idx = df_escolarizacion_idx.rename(columns={"code":"code_un", 
                                                              "average_total_years_of_schooling_for_adult_population":"anios_escolarizacion",})

df_escolarizacion_idx = clean_country_names(df=df_escolarizacion_idx, column="entity")
df_escolarizacion_idx.year = df_escolarizacion_idx.year.astype(int)
df_escolarizacion_idx.head()

,country,code_un,year,anios_escolarizacion
0,afghanistan,AFG,1870,0.01
1,afghanistan,AFG,1875,0.01
2,afghanistan,AFG,1880,0.01
3,afghanistan,AFG,1885,0.01
4,afghanistan,AFG,1890,0.01


In [102]:
df_base = df_base.merge(right=df_escolarizacion_idx, on=["country", "year"], how="inner")
print("paises: ", len(df_base.country.unique()))
print("filas: ", df_base.shape)
df_base.head()

paises:  159
filas:  (795, 11)


,code,country,year,state_fragility_idx,free_not_free,democracy_idx,code_wb,tasa_pob_activa,esperanza_vida,code_un,anios_escolarizacion
0,AFG,afghanistan,2013,21,NF,27,AFG,48.016998,62.525,AFG,3.5
1,AFG,afghanistan,2014,21,NF,26,AFG,48.188999,62.966,AFG,3.5
2,AFG,afghanistan,2015,21,NF,24,AFG,48.374001,63.377,AFG,3.6
3,AFG,afghanistan,2016,21,NF,24,AFG,48.516998,63.763,AFG,3.6
4,AFG,afghanistan,2017,21,NF,24,AFG,48.666000,64.130,AFG,3.8


### Sistema electoral

In [103]:
df_sistemaselectorales_idx = df_sistemaselectorales
df_sistemaselectorales_idx = clean_country_names(df=df_sistemaselectorales_idx, column="country")
df_sistemaselectorales_idx.year = df_sistemaselectorales_idx.year.astype(int)
df_sistemaselectorales_idx = df_sistemaselectorales_idx.sort_values(["country", "year"], ascending=False).groupby(["country"]).head(1)
df_sistemaselectorales_idx = df_sistemaselectorales_idx.drop(columns=["year"])
df_sistemaselectorales_idx.head()

,country,electoral_system_family,electoral_system_for_national_legislature,number_of_tiers,legislative_size_directly_elected,legislative_size_voting_members,electoral_system_for_the_president
1345,zimbabwe,Mixed,Parallel,2,270.0,270.0,TRS
1339,zambia,Plurality/Majority,FPTP,1,156.0,165.0,TRS
1336,yemen,Plurality/Majority,FPTP,1,301.0,301.0,TRS
1332,virgin_islands_british,Plurality/Majority,FPTP,2,13.0,13.0,Not applicable
1326,vietnam,Plurality/Majority,TRS,1,496.0,496.0,Not applicable


In [104]:
df_base = df_base.merge(right=df_sistemaselectorales_idx, on=["country"], how="inner")
print("paises: ", len(df_base.country.unique()))
print("filas: ", df_base.shape)
df_base.head()

paises:  159
filas:  (795, 17)


,code,country,year,state_fragility_idx,free_not_free,democracy_idx,code_wb,tasa_pob_activa,esperanza_vida,code_un,anios_escolarizacion,electoral_system_family,electoral_system_for_national_legislature,number_of_tiers,legislative_size_directly_elected,legislative_size_voting_members,electoral_system_for_the_president
0,AFG,afghanistan,2013,21,NF,27,AFG,48.016998,62.525,AFG,3.5,Other,SNTV,1,250.0,250.0,TRS
1,AFG,afghanistan,2014,21,NF,26,AFG,48.188999,62.966,AFG,3.5,Other,SNTV,1,250.0,250.0,TRS
2,AFG,afghanistan,2015,21,NF,24,AFG,48.374001,63.377,AFG,3.6,Other,SNTV,1,250.0,250.0,TRS
3,AFG,afghanistan,2016,21,NF,24,AFG,48.516998,63.763,AFG,3.6,Other,SNTV,1,250.0,250.0,TRS
4,AFG,afghanistan,2017,21,NF,24,AFG,48.666000,64.130,AFG,3.8,Other,SNTV,1,250.0,250.0,TRS


### Población

In [105]:
df_poblacion_idx = pd.melt(df_poblacion, id_vars=['country_name','country_code'], value_vars=df_poblacion.columns[4:])
df_poblacion_idx = df_poblacion_idx[["country_code", "variable", "value"]]
df_poblacion_idx = df_poblacion_idx.rename(columns={"country_code":"code_wb", "variable":"year",
                                                         "value":"poblacion"})
df_poblacion_idx.year = df_poblacion_idx.year.astype(int)
df_poblacion_idx.head()

,code_wb,year,poblacion
0,ABW,1960,54211.0
1,AFG,1960,8996973.0
2,AGO,1960,5454933.0
3,ALB,1960,1608800.0
4,AND,1960,13411.0


In [106]:
df_base = df_base.merge(right=df_poblacion_idx, on=["code_wb", "year"], how="inner")
print("paises: ", len(df_base.country.unique()))
print("filas: ", df_base.shape)
df_base.head()

paises:  159
filas:  (795, 18)


,code,country,year,state_fragility_idx,free_not_free,democracy_idx,code_wb,tasa_pob_activa,esperanza_vida,code_un,anios_escolarizacion,electoral_system_family,electoral_system_for_national_legislature,number_of_tiers,legislative_size_directly_elected,legislative_size_voting_members,electoral_system_for_the_president,poblacion
0,AFG,afghanistan,2013,21,NF,27,AFG,48.016998,62.525,AFG,3.5,Other,SNTV,1,250.0,250.0,TRS,32269589.0
1,AFG,afghanistan,2014,21,NF,26,AFG,48.188999,62.966,AFG,3.5,Other,SNTV,1,250.0,250.0,TRS,33370794.0
2,AFG,afghanistan,2015,21,NF,24,AFG,48.374001,63.377,AFG,3.6,Other,SNTV,1,250.0,250.0,TRS,34413603.0
3,AFG,afghanistan,2016,21,NF,24,AFG,48.516998,63.763,AFG,3.6,Other,SNTV,1,250.0,250.0,TRS,35383128.0
4,AFG,afghanistan,2017,21,NF,24,AFG,48.666000,64.130,AFG,3.8,Other,SNTV,1,250.0,250.0,TRS,36296400.0


### Constituciones

In [107]:
df_cow = df_cow.rename(columns={"ccode":"cowcode"})
df_cow = df_cow[["stateabb", "cowcode", "statenme"]].drop_duplicates()

In [108]:
df_constituciones_idx = df_constituciones_red.merge(right=df_cow, on="cowcode", how="inner")

In [109]:
df_constituciones_idx = df_constituciones_idx.sort_values(["cowcode", "year"], ascending=False).groupby(["cowcode"]).head(1)

In [110]:
df_constituciones_idx = df_constituciones_idx.drop(columns=["cowcode", "country", "year"], axis=1)
df_constituciones_idx = df_constituciones_idx.rename(columns={"stateabb":"code"})

code_changes={"DRV":"VIE","YAR":"YEM","SSD":"SSU", "SUD":"SDN", "ETH":"ETI", "ROM":"RUM", }
df_constituciones_idx["code"] =  df_constituciones_idx["code"].replace(code_changes)
df_constituciones_idx.head()

,hoselect,hosterm,hosdiss,hogelect,hogterm,hogdiss,legisl,housenum,fedunit,code,statenme
11832,3.0,5,1.0,4.0,0,1.0,1.0,2.0,NaN,WSM,Samoa
9356,3.0,4,1.0,99.0,99,99.0,1.0,2.0,1.0,FSM,Federated States of Micronesia
10569,2.0,4,1.0,99.0,99,99.0,1.0,3.0,98.0,PAL,Palau
9014,3.0,0,1.0,99.0,99,99.0,1.0,2.0,NaN,MSI,Marshall Islands
7636,2.0,0,1.0,99.0,99,99.0,NaN,NaN,NaN,NAU,Nauru


In [111]:
df_base = df_base.merge(right=df_constituciones_idx, on=["code"], how="inner")
print("paises: ", len(df_base.country.unique()))
print("filas: ", df_base.shape)
df_base.head()

paises:  155
filas:  (775, 28)


,code,country,year,state_fragility_idx,free_not_free,democracy_idx,code_wb,tasa_pob_activa,esperanza_vida,code_un,anios_escolarizacion,electoral_system_family,electoral_system_for_national_legislature,number_of_tiers,legislative_size_directly_elected,legislative_size_voting_members,electoral_system_for_the_president,poblacion,hoselect,hosterm,hosdiss,hogelect,hogterm,hogdiss,legisl,housenum,fedunit,statenme
0,AFG,afghanistan,2013,21,NF,27,AFG,48.016998,62.525,AFG,3.5,Other,SNTV,1,250.0,250.0,TRS,32269589.0,2.0,5,1.0,99.0,99,99.0,1.0,3.0,NaN,Afghanistan
1,AFG,afghanistan,2014,21,NF,26,AFG,48.188999,62.966,AFG,3.5,Other,SNTV,1,250.0,250.0,TRS,33370794.0,2.0,5,1.0,99.0,99,99.0,1.0,3.0,NaN,Afghanistan
2,AFG,afghanistan,2015,21,NF,24,AFG,48.374001,63.377,AFG,3.6,Other,SNTV,1,250.0,250.0,TRS,34413603.0,2.0,5,1.0,99.0,99,99.0,1.0,3.0,NaN,Afghanistan
3,AFG,afghanistan,2016,21,NF,24,AFG,48.516998,63.763,AFG,3.6,Other,SNTV,1,250.0,250.0,TRS,35383128.0,2.0,5,1.0,99.0,99,99.0,1.0,3.0,NaN,Afghanistan
4,AFG,afghanistan,2017,21,NF,24,AFG,48.666000,64.130,AFG,3.8,Other,SNTV,1,250.0,250.0,TRS,36296400.0,2.0,5,1.0,99.0,99,99.0,1.0,3.0,NaN,Afghanistan


In [ ]:
df_base